# Semana 06

## PROG: **`indcpa.py`**

Ilustre a codificação e animação do modelo de segurança **IND-CPA**
 com: 1. Cifra Identidade 2. Cifra ChaCha20 Como adversários deverá
 considerar em 1) a estratégia óbvia para atacar a cifra e, em 2), a
 estratégia que dá a resposta aleatória.

In [40]:
import os
import random
from abc import ABC, abstractmethod
from cryptography.hazmat.primitives.ciphers import Cipher as CryptoCipher, algorithms

In [41]:
class Cipher(ABC):
    @abstractmethod
    def keygen(self) -> bytes:
        pass
  
    @abstractmethod
    def enc(self, key: bytes, text: bytes) -> bytes:
        pass

    @abstractmethod
    def dec(self, key: bytes, ciphertext: bytes) -> bytes:
        pass

class INDCPA_Adv(ABC):
    @abstractmethod
    def choose(self, oracle: callable) -> bytes:
        pass

    @abstractmethod
    def guess(self, oracle: callable, ciphertext: bytes) -> bool:
        pass

In [42]:
# C is the scheme (cipher)
# A is the adversary
def IND_CPA(C: Cipher, A: INDCPA_Adv) -> bool:
    k = C.keygen()
    enc_oracle = lambda ptxt: C.enc(k, ptxt)
    
    # adversário escolhe duas mensagens
    m0, m1 = A.choose(enc_oracle)
    assert len(m0) == len(m1), "As mensagens devem ter o mesmo tamanho"
    
    # atira a moeda
    b = random.choice([0, 1])
    c = C.enc(k, m0 if b == 0 else m1)
    
    # adversário tenta adivinhar qual foi cifrada
    b_prime = A.guess(enc_oracle, c)
    return b == b_prime

def calcular_vantagem(C: Cipher, A: INDCPA_Adv, iteracoes=10000):
    vitorias = sum(1 for _ in range(iteracoes) if IND_CPA(C, A))
    probabilidade = vitorias / iteracoes
    vantagem = 2 * abs(probabilidade - 0.5)
    print(f"Vitórias: {vitorias}/{iteracoes} | Vantagem: {vantagem:.4f}")
    return vantagem

In [39]:
class IdentityCipher(Cipher):
    def keygen(self): return b''
    def enc(self, key, text): return text
    def dec(self, key, ciphertext): return ciphertext

class IdentityAdv(INDCPA_Adv):
    def choose(self, oracle):
        return b"MENSAGEM_0", b"MENSAGEM_1"

    def guess(self, oracle, ciphertext):
        # ganha sempre
        return 0 if ciphertext == b"MENSAGEM_0" else 1

class SecureChaCha20(Cipher):
    def keygen(self): 
        return os.urandom(32) # chave de 256 bits
        
    def enc(self, key, text):
        nonce = os.urandom(16) 
        cipher = CryptoCipher(algorithms.ChaCha20(key, nonce), mode=None)
        return nonce + cipher.encryptor().update(text)
        
    def dec(self, key, ciphertext):
        nonce, actual_ciphertext = ciphertext[:16], ciphertext[16:]
        cipher = CryptoCipher(algorithms.ChaCha20(key, nonce), mode=None)
        return cipher.decryptor().update(actual_ciphertext)

class RandomAdv(INDCPA_Adv):
    def choose(self, oracle):
        return b"MENSAGEM_0", b"MENSAGEM_1"

    def guess(self, oracle, ciphertext):
        return random.choice([0, 1])

**Testes:**

In [34]:
print("Teste 1: Cifra Identidade")
calcular_vantagem(IdentityCipher(), IdentityAdv(), iteracoes=5000)

print("\nTeste 2: ChaCha20 Segura")
calcular_vantagem(SecureChaCha20(), RandomAdv(), iteracoes=5000)

Teste 1: Cifra Identidade
Vitórias: 5000/5000 | Vantagem: 1.0000

Teste 2: ChaCha20 Segura
Vitórias: 2493/5000 | Vantagem: 0.0028


0.0028000000000000247

### Explicação

**Teste 1: A Cifra Identidade**
- A Cifra (`IdentityCipher`): É uma cifra falsa. O que entra, sai exatamente igual. Se lhe deres a palavra "RICARDO", ela devolve "RICARDO".

- O Adversário (`IdentityAdv`): Manda duas mensagens diferentes ("MENSAGEM_0" e "MENSAGEM_1"). Quando recebe o desafio de volta, como a cifra não escondeu nada, ele lê o texto em claro e sabe instantaneamente qual foi a escolhida.

- O Resultado Esperado: O adversário ganha 100% das vezes. A vantagem é 1.0. Prova que a cifra é inútil.

**Teste 2: ChaCha20 Segura**
- A Cifra (`SecureChaCha20`): É uma cifra a sério. Para além da chave secreta, usa um Nonce (um número aleatório) diferente sempre que cifra alguma coisa. Mesmo que cifres a mesma palavra duas vezes, o criptograma sai completamente diferente.

- O Adversário (`RandomAdv`): Ele manda as duas mensagens. Quando recebe o desafio de volta, olha para o criptograma e não vê qualquer padrão. O oráculo não o ajuda porque cada vez que ele pede para cifrar algo, o resultado muda (devido ao novo *nonce*). Resta-lhe atirar a moeda ao ar e adivinhar.

- O Resultado Esperado: O adversário ganha cerca de 50% das vezes (pura sorte). A vantagem será muito próxima de 0.0. Prova que a cifra é segura no modelo IND-CPA.

## PROG: **`indcpa_attck.py`**

Um segundo passo consiste em considerar algumas das construções
envolvedo cifras simétricas que se estudarem, e que foram consideradas
inseguras. Ao invés de pedir que implementem todos os ataques
mencionados, sugere-se em vez disso implementem apenas o ponto `GRUPO%3+1`.

Sugestões:
 1. Insegurança no modelo **IND** do modo ECB de uma cifra por blocos (mensagens com mais do que um bloco)
 1. Insegurança no modelo **IND-CPA** do modo ECB de uma cifra por blocos, onde um único bloco é cifrado;
 1. Insegurança no modelo **IND-CPA** da cifra `ChaCha20` com *Nonce* fixo;

In [35]:
class BadChaCha20(Cipher):
    def __init__(self):
        self.fixed_nonce = b'\x00' * 16 

    def keygen(self):
        return os.urandom(32)
        
    def enc(self, key, text):
        cipher = CryptoCipher(algorithms.ChaCha20(key, self.fixed_nonce), mode=None)
        return cipher.encryptor().update(text)
        
    def dec(self, key, ciphertext): 
        pass 

class FixedNonceAdv(INDCPA_Adv):
    def choose(self, oracle):
        self.m0 = b"ATACAR_HOJE!"
        self.m1 = b"ATACAR_AMANH"
        return self.m0, self.m1

    def guess(self, oracle, ciphertext):
        c_m0 = oracle(self.m0)
        
        if ciphertext == c_m0:
            return 0
        return 1

**Teste:**

In [36]:
print("Teste 3: Ataque ChaCha20 Nonce Fixo")
calcular_vantagem(BadChaCha20(), FixedNonceAdv(), iteracoes=5000)

Teste 3: Ataque ChaCha20 Nonce Fixo
Vitórias: 5000/5000 | Vantagem: 1.0000


1.0

### Explicação

**Teste 3: ChaCha20 com Nonce Fixo**
- A Cifra (`BadChaCha20`): Aqui introduzimos uma falha fatal propositada. A cifra ChaCha20 está a usar um Nonce Fixo (sempre zeros) para todas as operações.

- O Adversário (`FixedNonceAdv`): Este adversário é inteligente. Ele escolhe as duas mensagens (m0 e m1). A seguir, ele usa o oráculo que o jogo lhe deu e pede: "Cifra-me a m0".

    - Como a chave é a mesma e o nonce está fixo (não muda), a operação matemática vai dar exatamente o mesmo resultado sempre.

    - Quando o jogo lhe entrega o criptograma do desafio, ele só tem de comparar: "Isto é igual ao criptograma da m0 que eu acabei de gerar no oráculo?". Se for igual, ele sabe que o jogo escolheu a m0 (0). Se for diferente, foi a m1 (1).

- O Resultado Esperado: O adversário quebra o sistema e ganha 100% das vezes. A vantagem é 1.0. Prova que reutilizar nonces numa cifra stream destrói completamente a confidencialidade.

## Outros Modelos

Para além dos modelos **IND** e **IND-CPA**, foram estudados outros
modelos de segurança, como **IND-CCA**, **INT-PTXT**,
**INT-CTXT**. Desta vez, sugere-se que escolham algumas das seguestões
abaixo apresentadas para implementar os ataques que ilustrem os
problemas de segurança das diferentes construções nesse modelos.

Sugestões:

 1. Insegurança **IND-CCA** do modo `CTR` de numa cifra de blocos;
 1. Insegurança **IND-PTXT** de se usar um simples *hash* (sem segredo) como *MAC*;
 2. Insegurança **IND-CPA** da combinação **encrypt & MAC**.
 1. Modo `CBC` de uma cifra por blocos com um `IV` implementado como um contador;

### **IND-CCA** do modo `CTR` de numa cifra de blocos

In [37]:
class INDCCA_Adv(ABC):
    @abstractmethod
    def choose(self, enc_oracle: callable) -> tuple[bytes, bytes]: 
        pass
    
    @abstractmethod
    def guess(self, enc_oracle: callable, dec_oracle: callable, ciphertext: bytes) -> bool: 
        pass

def IND_CCA(C: Cipher, A: INDCCA_Adv) -> bool:
    k = C.keygen()
    enc_oracle = lambda ptxt: C.enc(k, ptxt)
    
    # Oráculo de decifragem real
    def dec_oracle(ctxt: bytes, challenge_ctxt: bytes = None):
        if ctxt == challenge_ctxt:
            raise ValueError("Batota! Não podes pedir para decifrar o próprio desafio.")
        return C.dec(k, ctxt)
    
    m0, m1 = A.choose(enc_oracle)
    assert len(m0) == len(m1), "As mensagens devem ter o mesmo tamanho"
    
    b = random.choice([0, 1])
    challenge_c = C.enc(k, m0 if b == 0 else m1)
    
    # proteger o oracle
    safe_dec_oracle = lambda ctxt: dec_oracle(ctxt, challenge_c)
    
    b_prime = A.guess(enc_oracle, safe_dec_oracle, challenge_c)
    return b == b_prime

def calcular_vantagem_cca(C: Cipher, A: INDCCA_Adv, iteracoes=5000):
    vitorias = sum(1 for _ in range(iteracoes) if IND_CCA(C, A))
    probabilidade = vitorias / iteracoes
    vantagem = 2 * abs(probabilidade - 0.5)
    print(f"Vitórias: {vitorias}/{iteracoes} | Vantagem: {vantagem:.4f}")
    return vantagem

class AES_CTR_Cipher(Cipher):
    def keygen(self):
        return os.urandom(32)
        
    def enc(self, key, text):
        nonce = os.urandom(16)
        cipher = CryptoCipher(algorithms.AES(key), modes.CTR(nonce))
        return nonce + cipher.encryptor().update(text)
        
    def dec(self, key, ciphertext):
        nonce, actual_ctxt = ciphertext[:16], ciphertext[16:]
        cipher = CryptoCipher(algorithms.AES(key), modes.CTR(nonce))
        return cipher.decryptor().update(actual_ctxt)

class MalleableAdv(INDCCA_Adv):
    def choose(self, enc_oracle):
        # as mensagens agora começam com letras diferentes
        self.m0 = b"A_MENSAGEM"
        self.m1 = b"B_MENSAGEM"
        return self.m0, self.m1

    def guess(self, enc_oracle, dec_oracle, ciphertext):
        # separar o nonce do criptograma real
        nonce = ciphertext[:16]
        c_real = bytearray(ciphertext[16:])
        
        # alterar o primeiro byte do criptograma
        c_real[0] ^= 0x01
        
        # reconstruir o criptograma forjado
        forged_ciphertext = nonce + bytes(c_real)
        
        # pedir ao oráculo para decifrar.
        m_linha = dec_oracle(forged_ciphertext)
         
        # o resultado será exclusivo para a m0.
        expected_m0_first_byte = self.m0[0] ^ 0x01
        
        if m_linha[0] == expected_m0_first_byte:
            return 0
        return 1

**Teste:**

In [38]:
print("\nTeste 4: Ataque IND-CCA ao AES-CTR")
calcular_vantagem_cca(AES_CTR_Cipher(), MalleableAdv(), iteracoes=5000)


Teste 4: Ataque IND-CCA ao AES-CTR
Vitórias: 5000/5000 | Vantagem: 1.0000


1.0

### Explicação

No modo CTR, a cifra de blocos (AES) não cifra a mensagem diretamente. Em vez disso, cifra um contador para gerar uma *keystream* (uma sequência de bits pseudoaleatórios). Depois, faz-se um simples **XOR** bit-a-bit entre essa *keystream* e o texto limpo para obter o criptograma. 

A falha fatal da operação XOR é que, se um atacante inverter (fizer *flip*) a um bit no criptograma, **o bit correspondente no texto decifrado também será invertido**, e a cifra não tem forma de detetar que o ficheiro foi manipulado.

O nosso adversário tira partido do Oráculo de Decifragem usando a seguinte tática:

1. **Escolha Estratégica:** Escolhe duas mensagens que começam com letras **diferentes** (`A_MENSAGEM` e `B_MENSAGEM`). Isto é crucial para conseguir distingui-las mais tarde.
2. **Forjar um Criptograma:** Recebe o criptograma do desafio, vai ao primeiro byte (a primeira letra) e faz um XOR com `0x01` (inverte o último bit dessa letra).
3. **Contornar a Regra:** O modelo IND-CCA proíbe pedir para decifrar o *próprio* desafio. Mas como o adversário alterou um bit, este criptograma forjado é tecnicamente "novo". O oráculo de decifragem aceita-o e decifra-o.
4. **Verificar o Estrago:** O oráculo devolve a mensagem decifrada. Por causa da maleabilidade, a primeira letra estará alterada (será a letra original XOR `0x01`).
5. **A Vitória:** O adversário só tem de calcular o que aconteceria à letra "A" da primeira mensagem se sofresse aquele XOR. Se o resultado do oráculo bater certo com esse cálculo, ele tem a certeza absoluta de que o desafio escondia a Mensagem 0. Caso contrário, era a Mensagem 1.